Import libraries and functions

In [3]:
import numpy as np
import torch
import pyro
from torch import nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from codebase_0 import (
    CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
)
from poisson_bcann_hmc_nuts import (
    stratified_subsample, make_chain_inits, run_nuts, time_nuts, get_device, to_device
)
from gamma_bcann_vi import GammaBCANN, gamma_build_bcann_guide, gamma_fit_glm_mle
from bcann_neutra import neutra_fit_guide, time_neutra, run_neutra_parallel
from gamma_horseshoe import HorseshoeGammaBCANN



seed = 42
set_seed(seed)
pyro.clear_param_store()

Loading in data and train test split

In [ ]:
device = torch.device("cuda:3")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 10000
)

Initiate IAF, HorseshoeBCANN, and train the IAF with guide function

In [5]:
network_architecture = [20, 15, 10]

# flow (step 2)
NUM_TRANSFORMS = 2        # IAF layers; 0 -> diagonal Gaussian (NeuTra only rescales the space)
FLOW_HIDDEN = None       # IAF hidden widths; None -> [latent_dim]. Try [512] if the flow is slow
SVI_BATCH = None          # None -> full-batch ELBO on the subsample
SVI_STEPS = 5000
SVI_LR, SVI_LR_FINAL = 5e-3, 5e-4

glm_mle = gamma_fit_glm_mle(X_learn, y_learn, w_learn, random_state=seed)

with torch.device(device):
        horseshoe_gamma_bcann = HorseshoeGammaBCANN(
                in_features= 48,
                neural_net_params={"hidden_activation_nodes": network_architecture,
                                    "hidden_activations": nn.Tanh},
            )
        horseshoe_gamma_guide, horseshoe_gamma_history = neutra_fit_guide(
            horseshoe_gamma_bcann, 
            X_learn, 
            y_learn, 
            w_learn, 
            glm_mle=tuple(t.to(device) for t in glm_mle),
            num_transforms=NUM_TRANSFORMS, hidden_dim= FLOW_HIDDEN,
            num_steps=SVI_STEPS, lr=SVI_LR, lr_final=SVI_LR_FINAL, batch_size=10000,
            report_every=max(SVI_STEPS // 20, 1), random_state=seed
        )


step 250 | -ELBO/n: 9.3412 | 27s
step 500 | -ELBO/n: 8.6824 | 52s
step 750 | -ELBO/n: 9.8906 | 73s
step 1000 | -ELBO/n: 24.3561 | 88s
step 1250 | -ELBO/n: 8.6983 | 103s
step 1500 | -ELBO/n: 183.7850 | 117s
step 1750 | -ELBO/n: 8.6676 | 131s
step 2000 | -ELBO/n: 9.1062 | 144s
step 2250 | -ELBO/n: 8.6585 | 158s
step 2500 | -ELBO/n: 8.6500 | 172s
step 2750 | -ELBO/n: 8.7504 | 185s
step 3000 | -ELBO/n: 108.5556 | 200s
step 3250 | -ELBO/n: 61.2894 | 214s
step 3500 | -ELBO/n: 8.7705 | 229s
step 3750 | -ELBO/n: 8.6428 | 247s
step 4000 | -ELBO/n: 9.3159 | 263s
step 4250 | -ELBO/n: 8.7133 | 277s
step 4500 | -ELBO/n: 22.6555 | 291s
step 4750 | -ELBO/n: 2368.0252 | 304s
step 5000 | -ELBO/n: 84.1805 | 319s


Initiate parameters for NUTS

In [10]:
# NUTS (step 3)
CHAINS = 6
WARMUP, SAMPLES = 1500, 1500
TARGET_ACCEPT = 0.9
MAX_TREE_DEPTH = 8

nuts_kwargs = dict(num_warmup=WARMUP, num_samples=SAMPLES, target_accept=TARGET_ACCEPT,
                       max_tree_depth=MAX_TREE_DEPTH, init_scale=1.0, jit_compile = True)


Horseshoe test run

In [7]:
time_neutra(
    horseshoe_gamma_bcann, horseshoe_gamma_guide, 
    X_train, 
    y_train,
    w_train,
    n_iter=20,
    device=device
)  # cost estimate first

6.02 s / iteration on cuda:3  (early warmup iterations are not representative of tree depth)


6.0206902623176575

Horseshoe full NeuTra run

In [11]:
gamma_horseshoe_neutra_samples, gamma_horseshoe_neutra_info = run_neutra_parallel(
    HorseshoeGammaBCANN, 
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ), 
    horseshoe_gamma_guide, 
    X_train, 
    y_train,
    w_train,
    num_chains=CHAINS, devices=device,
    out_dir=f"neutra_chains_gamma_horseshoe", seed=seed,
    **nuts_kwargs
)

elapsed 3:54:26
chain 5: Sample: 100%|██████████| 3000/3000 [3:51:53,  4.64s/it, step size=4.30e-04, acc. prob=0.878]


Write to disk

In [12]:
torch.save(
    {
        "samples": gamma_horseshoe_neutra_samples,
        "info": gamma_horseshoe_neutra_info,
        "config": dict(num_warmup=WARMUP, num_samples=SAMPLES, target_accept=TARGET_ACCEPT,
                       max_tree_depth=MAX_TREE_DEPTH, seed=seed, n_chains=6),
    },
    "hmc_results/gamma_horseshoe_neutra_full_set.pt",
)
